# 02 — The agent loop
A model call returns text. An agent can choose a tool, inspect its result,
and continue. This tiny Python loop makes that mechanism visible first.



## Problem

A one-shot answer cannot request and inspect new evidence.


## Concept

An agent loop alternates model decisions and observations until a stop condition.

An agent adds a feedback loop around the model: propose an action, run it through the harness, append the observation, then decide whether to continue. The model requests; the runtime executes.


## Abstraction

task → model → action? → tool → observation → model → answer. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
task ──▶ Agent / LLM ──▶ choose action? ──yes──▶ tool
             ▲                                   │        │
             │                              observation ◀┘
             └──────── repeat until answer or max_iter ──────────
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Trace action, observation, and stop in plain Python first

Trace action, observation, and stop in plain Python first.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task

from workshop.common import model
from workshop.tools import calculate_annual_roi

# A deterministic stand-in for the model's action decision. It lets us see the
# loop before CrewAI and the model take over the decision in the next cell.
state: dict[str, float | None] = {"roi": None}
for step in range(3):
    if state["roi"] is None:
        print(f"Step {step + 1}: choose calculate_annual_roi")
        state["roi"] = calculate_annual_roi(45_000, 30_000)
        print(f"Observation: {state['roi']}% scenario ROI")
    else:
        print(f"Step {step + 1}: answer using observation, with a caveat")
        break



## A bounded CrewAI agent
CrewAI now manages the model/tool loop. The role, task, and limits are explicit.
There is one agent and one task, so this is the smallest useful CrewAI run.



## Step 2 — Build a bounded CrewAI Agent and Task

Build a bounded CrewAI Agent and Task; inspect the final output as one run of the loop.


In [ ]:
researcher = Agent(
    role="Enterprise AI Researcher",
    goal="Explain an adoption question without pretending unknowns are facts",
    backstory="You are cautious about weak evidence.",
    llm=model(),
    max_iter=3,
    max_execution_time=90,
    allow_delegation=False,
)
task = Task(
    description="Should Northstar pilot Meridian AI? We estimate $45k annual benefit "
    "and $30k cost, but have not verified data deletion or throughput.",
    expected_output="A short conditional recommendation and two unknowns.",
    agent=researcher,
)
result = Crew(agents=[researcher], tasks=[task], process=Process.sequential).kickoff()
print(result.raw)


## Execution flow

task → model → action? → tool → observation → model → answer. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

Without a stop bound, repeated tool choices consume time and tokens. `max_iter` and `max_execution_time` cap the CrewAI run; the deterministic loop makes that bound visible.


## Production implication

Set iteration, time, and cost limits for every run. Capture the reason a run stopped so timeout and success are not confused.


## Your turn

Find the maximum iteration setting and explain what it prevents.

**Checkpoint:** You can name the role, task, observation, and stopping bound.
